# NCDB: End-to-End Data Pipeline

**Run all cells once** to perform the complete project workflow:

1. **Aggregation:** download the National Collision Database years (starting with 2019–2017, extending back if necessary), clean codes, and aggregate to one collision per row.
2. **Exploratory Data Analysis (EDA):** visualize the intersection subset and compare intersection vs non-intersection collisions.
3. **Preprocessing:** prepare the modeling data, split train/test, encode features, select features, and evaluate PCA.

This notebook combines the latest supplied versions of `01_aggregation(7).ipynb`, `02_eda(3).ipynb`, and `03_preprocessing (2)(1).ipynb`. The original analysis steps and charts are retained. **No manual file uploads or CSV handoffs between stages are needed.**

**Before running:** Select a Python 3 kernel and choose **Run All** (Jupyter) or **Runtime → Run all** (Colab). An **internet connection is required on the first run** to download the public NCDB source workbooks and install any missing dependencies. Original yearly files are cached locally so reruns can reuse them. All generated CSVs plus a single ZIP archive are written to `outputs/`. This workflow may require significant RAM and processing time for large NCDB datasets.


In [ ]:
# One-time setup: install missing Python dependencies if necessary.
import importlib.util
import subprocess
import sys
from pathlib import Path

REQUIRED_PACKAGES = {
    "numpy": "numpy", "pandas": "pandas", "openpyxl": "openpyxl",
    "matplotlib": "matplotlib", "seaborn": "seaborn", "scipy": "scipy",
    "sklearn": "scikit-learn",
}
missing_packages = [package for module, package in REQUIRED_PACKAGES.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

OUTPUT_DIR = Path("outputs")
SOURCE_CACHE_DIR = Path("ncdb_source_cache")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
AUTO_DOWNLOAD_ZIP_IN_COLAB = False  # Set True to download the results ZIP at the end in Colab.
print("Output directory:", OUTPUT_DIR.resolve())


---

# 01 — Load and aggregate collisions

*Source: `01_aggregation(7).ipynb`*

# Step 1: Load and Aggregate NCDB (2017–2019, extend if needed)

**Input:** NCDB annual files (one row per person involved in a collision)
**Output:** `outputs/ncdb_<years>_collisions.csv` (one row per collision, with features and target `Casualty_Class`)

Code meanings come from the NCDB Data Dictionary:
https://open.canada.ca/data/dataset/1eb9eba7-71d1-4b30-9fb1-30cbdab7e63a/resource/21eb7966-38da-4814-a80e-521bce6c4c27/download/data_dictionary.pdf

## 1. Load the raw data
Loads data from 2019 to 2017. If observations are not enough, extends to earlier years as well

In [ ]:
import pandas as pd
import numpy as np

BASE = "https://open.canada.ca/data/dataset/1eb9eba7-71d1-4b30-9fb1-30cbdab7e63a/resource/"
SOURCES = {
    2019: "91d586f9-fb72-4eac-b1f5-d9b95830c877",
    2018: "bf937c33-6fc2-4010-8036-d91d4e1a6285",
    2017: "1e384c9e-6af2-40b3-9457-719fbce819c0",
    2016: "552280a5-f030-41c5-ac7d-649e42f2cb50",
    2015: "cbee1eb5-c853-47fa-b807-0cee9ecd286d",
    2014: "ea6471b8-2924-419b-9ade-baa8d68441ab",
    2013: "c0ea4b35-353c-40fc-9b2b-cd99b5b58801",
    2012: "cd77a978-f506-406e-afd1-b1e61e77dd9a",
    2011: "3fefd7a0-3c49-4eef-97e0-b5310be0fb07",
    2010: "53018df0-82d2-4c9a-8f83-e8f75b3a4300",
    2009: "21b9ce1c-4a83-41fa-ab02-c45b3747c362",
    2008: "60562270-5ffd-4dd7-a7ea-3461bcf245cb",
    2007: "b0366cbf-1400-4b9b-9bea-9ee7ad468b3d",
    2006: "c3c61c6f-706a-41f4-b0c8-bf1e6aa46a8b",
    2005: "d83abcbf-a61e-4fa0-b59a-dfdb4cae4bf8",
    2004: "da00a00b-3355-47f2-b313-58afa7314b15",
    2003: "213ba39d-02cd-43b5-a442-f5f9b80f9d13",
    2002: "b9bafa34-5d55-4353-81b6-f0941ff33bc0",
    2001: "9f696115-84a6-4425-b679-28f677df7c50",
    2000: "d6de0f0f-fad0-4e53-a62a-7024196203d0",
    1999: "80c765bc-eabe-4cc8-86f9-e464cd47e091",
}

FEATURE_COLS = ["C_RCFG", "C_TRAF", "C_WTHR", "C_RSUR", "C_RALN", "C_CONF",
                "C_HOUR", "C_WDAY", "C_MNTH", "C_VEHS"]
SAFETY_KNOWN = {"01", "02", "09", "10", "11", "12", "13"}


def count_usable(part):
    x = part[["C_YEAR", "C_CASE", *FEATURE_COLS, "P_ISEV", "P_SAFE"]].apply(lambda s: s.str.strip())
    x = x.mask(x.apply(lambda s: s.str.fullmatch(r"[UX]+", na=False)))
    for c in ("C_RCFG", "C_TRAF", "C_CONF", "P_SAFE"):
        x[c] = x[c].str.replace(r"^(\d)$", r"0\1", regex=True)
    x["casualty"] = x["P_ISEV"].isin(["2", "3"])
    x["known_safety"] = x["P_SAFE"].isin(SAFETY_KNOWN)
    x["no_safety"] = x["P_SAFE"].isin(["01", "13"])
    g = x.groupby(["C_YEAR", "C_CASE"])
    a = g.agg({**{c: "first" for c in FEATURE_COLS},
               "casualty": "max", "known_safety": "max", "no_safety": "max"})
    for c in ("C_HOUR", "C_WDAY", "C_MNTH", "C_VEHS"):
        a[c] = pd.to_numeric(a[c], errors="coerce")
    valid = (a["C_RCFG"].isin(["02", "03", "09"]) & a["casualty"]
             & a[FEATURE_COLS].notna().all(axis=1) & a["known_safety"])
    return int(valid.sum())


URLS, frames, usable = {}, [], 0
for year in range(2019, 1998, -1):
    url = BASE + f"{SOURCES[year]}/download/y_{year}_en.xlsx"
    cached_file = SOURCE_CACHE_DIR / f"y_{year}_en.xlsx"
    if not cached_file.exists():
        from urllib.request import urlretrieve
        partial = cached_file.with_suffix(".xlsx.part")
        try:
            print(f"Downloading {year} NCDB workbook...")
            urlretrieve(url, partial)
            partial.replace(cached_file)
        except Exception as exc:
            partial.unlink(missing_ok=True)
            raise RuntimeError(
                f"Could not download {year} NCDB workbook from {url}. "
                "Check that internet access is enabled, or put the original "
                f"workbook at {cached_file}."
            ) from exc
    part = pd.read_excel(cached_file, dtype=str)
    part.columns = part.columns.str.strip().str.upper()
    assert part["C_YEAR"].str.strip().eq(str(year)).all(), f"Unexpected year in {year} data"
    year_usable = count_usable(part)
    usable += year_usable
    URLS[year] = url
    frames.append(part)
    print(f"{year}: {year_usable:,} usable intersection collisions; total {usable:,}")
    if year <= 2017 and usable >= 50_000:
        break

assert usable >= 50_000, "Fewer than 50,000 usable intersection collisions in 1999–2019"
raw = pd.concat(frames, ignore_index=True)
print(f"Loaded {min(URLS)}–2019; {usable:,} intersection complete cases")
raw.head()

In [ ]:
EXPECTED = ["C_CASE", "C_YEAR", "C_MNTH", "C_WDAY", "C_HOUR", "C_SEV", "C_VEHS",
            "C_CONF", "C_RCFG", "C_WTHR", "C_RSUR", "C_RALN", "C_TRAF",
            "V_ID", "V_TYPE", "V_YEAR", "P_ID", "P_SEX", "P_AGE", "P_PSN",
            "P_ISEV", "P_SAFE", "P_USER"]
            
assert all(set(part.columns) == set(EXPECTED) for part in frames), "Unexpected NCDB columns"
assert raw[["C_YEAR", "C_CASE"]].notna().all().all(), "Missing year or collision ID"
print("Column validation passed for all loaded years")

## 2. Convert NCDB missing codes

NCDB uses letter codes instead of blanks:
- `U` / `UU` / `UUUU` = unknown, `X` / `XX` = not provided by jurisdiction -> treated as **missing (NaN)**
- `Q` / `QQ` = "other" -> kept as its own category
- `N` / `NN` = not applicable -> kept as its own category

In [ ]:
df = raw.copy()
for col in df.columns:
    df[col] = df[col].str.strip()

# Values made only of U and/or X characters are missing
missing_mask = df.apply(lambda s: s.str.fullmatch(r"[UX]+", na=False))

print("Missing codes per column (person level):")
summary = pd.DataFrame({
    "n_missing": missing_mask.sum(),
    "pct_missing": (missing_mask.mean() * 100).round(2),
}).sort_values("pct_missing", ascending=False)
display(summary)

df = df.mask(missing_mask)

# Some years store these codes without the leading zero ("2" instead of "02")
for c in ("C_RCFG", "C_TRAF", "C_CONF", "P_SAFE"):
    df[c] = df[c].str.replace(r"^(\d)$", r"0\1", regex=True)

## 3. Person-level flags

- `P_ISEV`: 1 = no injury, 2 = injury, 3 = fatality, N = not applicable (e.g. parked car placeholder)
- `P_USER`: 4 = bicyclist; `V_TYPE`: 17 = bicycle
- `V_TYPE` heavy vehicles: 07 unit trucks > 4536 kg, 08 road tractors, 09-11 buses
- `P_SAFE`: 01 = no safety device used, 13 = no device (not available)

In [ ]:
HEAVY_TYPES = {"07", "08", "09", "10", "11"}   # edit if you define "heavy" differently
NO_SAFETY = {"01", "13"}
SAFETY_KNOWN = {"01", "02", "09", "10", "11", "12", "13"}

df["is_injury"] = df["P_ISEV"].eq("2")
df["is_fatal"] = df["P_ISEV"].eq("3")
df["isev_unknown"] = df["P_ISEV"].isna()
df["is_cyclist"] = df["P_USER"].eq("4") | df["V_TYPE"].eq("17")
df["is_heavy"] = df["V_TYPE"].isin(HEAVY_TYPES)
df["no_safety"] = df["P_SAFE"].isin(NO_SAFETY)
df["safety_known"] = df["P_SAFE"].isin(SAFETY_KNOWN)

df[["is_injury", "is_fatal", "isev_unknown", "is_cyclist", "is_heavy", "no_safety"]].mean().round(4)

## 4. Aggregate to one row per collision

In [ ]:
COLLISION_COLS = ["C_YEAR", "C_MNTH", "C_WDAY", "C_HOUR", "C_SEV", "C_VEHS",
                  "C_CONF", "C_RCFG", "C_WTHR", "C_RSUR", "C_RALN", "C_TRAF"]

# Sanity check: collision-level fields should be the same for every person in a collision
n_unique = df.groupby(["C_YEAR", "C_CASE"])[COLLISION_COLS].nunique()
print("Collisions with inconsistent collision-level values:")
print((n_unique > 1).sum())

In [ ]:
agg_spec = {c: (c, "first") for c in COLLISION_COLS if c != "C_YEAR"}
agg_spec.update(
    Num_Persons=("P_ID", "size"),
    Total_Injuries=("is_injury", "sum"),
    Total_Fatalities=("is_fatal", "sum"),
    Unknown_Severity_Persons=("isev_unknown", "sum"),
    Heavy_Vehicle_Present=("is_heavy", "max"),
    Cyclist_Present=("is_cyclist", "max"),
    Any_No_Safety_Device=("no_safety", "max"),
    Safety_Known_Persons=("safety_known", "sum"),
)

col = df.groupby(["C_CASE", "C_YEAR"]).agg(**agg_spec).reset_index()

# Safety device flag: 1 if anyone had no device, 0 if all known people used one,
# NaN if nobody's safety device status is known
col["Safety_Device_Not_Used"] = np.where(
    col["Any_No_Safety_Device"], 1,
    np.where(col["Safety_Known_Persons"] > 0, 0, np.nan))
col = col.drop(columns=["Any_No_Safety_Device", "Safety_Known_Persons"])

for c in ["Heavy_Vehicle_Present", "Cyclist_Present"]:
    col[c] = col[c].astype(int)

print("Collisions:", len(col))
col.head()

## 5. Derived outcome variables and type conversion

In [ ]:
col["Total_Casualties"] = col["Total_Injuries"] + col["Total_Fatalities"]
col["Fatal_Collision"] = (col["Total_Fatalities"] > 0).astype(int)

col["Casualty_Class"] = pd.cut(col["Total_Casualties"], bins=[0, 1, 2, np.inf],
                               labels=["1", "2", "3+"], right=True)
# Collisions with 0 known casualties get NaN (should be rare: NCDB only has injury/fatal collisions)

for c in ["C_YEAR", "C_MNTH", "C_WDAY", "C_HOUR", "C_VEHS"]:
    col[c] = pd.to_numeric(col[c], errors="coerce")

# Intersection flag (C_RCFG 02 = public road intersection, 03 = driveway/parking lot entrance,
# 09 = traffic circle). Kept as a flag so you can decide later whether to filter.
col["Is_Intersection"] = col["C_RCFG"].isin(["02", "03", "09"]).astype(int)

col.dtypes

## 6. Sanity checks

In [ ]:
print("Collisions by year:\n", col.groupby("C_YEAR").size())
assert set(col["C_YEAR"]) == set(URLS)
assert not col[["C_YEAR", "C_CASE"]].duplicated().any()

print("Casualty_Class distribution (%):")
print((col["Casualty_Class"].value_counts(normalize=True, dropna=False) * 100).round(2))

print("\nZero-casualty collisions:", (col["Total_Casualties"] == 0).sum())
print("Collisions with >=1 person of unknown severity:", (col["Unknown_Severity_Persons"] > 0).sum())

# C_SEV: 1 = fatal collision, 2 = non-fatal injury collision
check = pd.crosstab(col["C_SEV"], col["Fatal_Collision"])
print("\nC_SEV vs our Fatal_Collision (should line up):")
print(check)

print("\nIntersection collisions:", col["Is_Intersection"].sum(),
      f"({col['Is_Intersection'].mean() * 100:.1f}%)")
# Final check: match the EDA definition of complete cases among intersections.
EDA_FEATURES = ["C_RCFG", "C_TRAF", "C_WTHR", "C_RSUR", "C_RALN", "C_CONF",
                "C_HOUR", "C_WDAY", "C_MNTH", "C_VEHS", "Heavy_Vehicle_Present",
                "Cyclist_Present", "Safety_Device_Not_Used"]
n_clean = len(col.loc[col["Is_Intersection"].eq(1)].dropna(subset=EDA_FEATURES + ["Casualty_Class"]))
print(f"Usable intersection collisions after aggregation and cleaning: {n_clean:,}")
assert n_clean == usable, f"Loading check ({usable:,}) != aggregated check ({n_clean:,})"
assert n_clean >= 50_000, "Insufficient intersection complete cases"

In [ ]:
print("Missing values per column (collision level, %):")
(col.isna().mean() * 100).round(2).sort_values(ascending=False)

## 7. Save

In [ ]:
years = sorted(col["C_YEAR"].dropna().astype(int).unique())
OUT = f"ncdb_{'_'.join(map(str, years))}_collisions.csv"
AGGREGATED_FILE = OUTPUT_DIR / OUT
col.to_csv(AGGREGATED_FILE, index=False)
# Keep the collision-level data in memory for BOTH later stages (no CSV reload needed).
aggregated_collisions = col.copy()
print(f"Saved {AGGREGATED_FILE}: {len(col):,} collisions from {', '.join(map(str, years))}")

---

# 02 — Exploratory data analysis

*Source: `02_eda(3).ipynb`*

# Step 2: Exploratory Data Analysis (EDA), intersection collisions only

**Input:** in-memory collision-level data from Step 1 (one row per collision, years selected during aggregation)

**Scope:** the project studies collisions at intersections, so all EDA below uses only collisions with
`Is_Intersection = 1` (road configuration 02 = intersection of public roads, 03 = driveway/parking lot entrance,
09 = traffic circle). Section 1b compares intersection and non-intersection collisions to justify this scope.

**Sections**
1. Load data and set up (and filter to intersections)
1b. Intersection vs non-intersection comparison
2. Structure and overview
3. Missing values
4. Target distribution (class imbalance)
5. Feature distributions
6. Outlier check
7. Feature vs target relationships
8. Correlation heatmaps

Code labels in section 1 are for chart readability. Verify them against the NCDB Data Dictionary.

## 1. Load data and set up

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

# Reuse Step 1's data, preserving leading-zero category codes.
col = aggregated_collisions.copy()

CLASS_ORDER = ["1", "2", "3+"]
col["Casualty_Class"] = pd.Categorical(col["Casualty_Class"].astype("string"),
                                       categories=CLASS_ORDER, ordered=True)
print("All collisions:", col.shape)

# Keep the full dataset only for the comparison in section 1b
col_all = col.copy()
col = col[col["Is_Intersection"] == 1].copy()
print(f"Intersection collisions: {len(col):,} of {len(col_all):,} ({len(col)/len(col_all)*100:.1f}%)")

In [ ]:
# Readable labels for charts (check against the data dictionary)
LABELS = {
    "C_WTHR": {"1": "Clear", "2": "Overcast", "3": "Rain", "4": "Snow",
               "5": "Freezing rain/hail", "6": "Low visibility", "7": "Strong wind", "Q": "Other"},
    "C_RSUR": {"1": "Dry", "2": "Wet", "3": "Snow", "4": "Slush", "5": "Icy",
               "6": "Sand/gravel", "7": "Muddy", "8": "Oil", "9": "Flooded", "Q": "Other"},
    "C_RALN": {"1": "Straight, level", "2": "Straight, gradient", "3": "Curved, level",
               "4": "Curved, gradient", "5": "Top of hill", "6": "Bottom of hill", "Q": "Other"},
    "C_RCFG": {"01": "Non-intersection", "02": "Intersection", "03": "Driveway/parking",
               "04": "Railroad crossing", "05": "Bridge/overpass", "06": "Tunnel",
               "07": "Passing lane", "08": "Ramp", "09": "Traffic circle",
               "10": "Express lane", "11": "Collector lane", "12": "Transfer lane", "QQ": "Other"},
    "C_TRAF": {"01": "Signals (working)", "02": "Signals (flashing)", "03": "Stop sign",
               "04": "Yield sign", "05": "Warning sign", "06": "Ped crosswalk",
               "07": "Police officer", "08": "School guard", "09": "School crossing",
               "10": "Reduced speed zone", "11": "No passing zone", "12": "Road markings",
               "13": "School bus", "15": "Rail signals/gates", "16": "Rail signs only",
               "17": "Not specified", "18": "No control", "QQ": "Other"},
    "C_CONF": {"01": "Hit moving object", "02": "Hit stationary object", "03": "Ran off left",
               "04": "Ran off right", "05": "Rollover", "06": "Other single vehicle",
               "21": "Rear-end", "22": "Side swipe", "23": "Left turn conflict (same dir)",
               "24": "Right turn conflict (same dir)", "25": "Other same direction",
               "31": "Head-on", "32": "Approaching side swipe", "33": "Left turn across traffic",
               "34": "Right turn", "35": "Right angle", "36": "Other different direction",
               "41": "Hit parked vehicle", "QQ": "Other"},
}

def label(s, name):
    """Map codes to readable labels; unknown codes keep their raw value."""
    return s.map(lambda v: LABELS[name].get(v, v) if pd.notna(v) else "Missing")

CAT_FEATURES = ["C_RCFG", "C_TRAF", "C_WTHR", "C_RSUR", "C_RALN", "C_CONF"]
NUM_FEATURES = ["C_HOUR", "C_WDAY", "C_MNTH", "C_VEHS"]
BIN_FEATURES = ["Heavy_Vehicle_Present", "Cyclist_Present", "Safety_Device_Not_Used"]
FEATURES = CAT_FEATURES + NUM_FEATURES + BIN_FEATURES   # the 13 candidate features
TARGET = "Casualty_Class"

# Built from the target: EDA only, never used as model inputs (data leakage)
LEAKAGE = ["Total_Injuries", "Total_Fatalities", "Total_Casualties", "Fatal_Collision", "C_SEV"]

## 1b. Intersection vs non-intersection collisions

Compares the casualty class mix of intersection collisions with all other collisions (non-intersection
and collisions with other or unknown road configuration). This motivates focusing the project on intersections.

In [ ]:
grp = np.where(col_all["Is_Intersection"] == 1, "Intersection", "Non-intersection / other")
comp = (pd.crosstab(grp, col_all[TARGET], normalize="index") * 100).reindex(columns=CLASS_ORDER).round(2)
comp["n"] = pd.Series(grp).value_counts().reindex(comp.index).values
display(comp)

ax = comp[CLASS_ORDER].T.plot.bar(figsize=(6.5, 4), rot=0, color=["#C44E52", "#8C8C8C"])
ax.set_xlabel("Casualties per collision"); ax.set_ylabel("% of collisions in group")
ax.set_title("Casualty_Class: intersection vs non-intersection")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=8)
ax.legend(title="")
plt.tight_layout(); plt.show()

## 2. Structure and overview

In [ ]:
print(f"Collisions: {len(col):,}   Columns: {col.shape[1]}")
overview = pd.DataFrame({
    "dtype": col[FEATURES + [TARGET]].dtypes.astype(str),
    "n_unique": col[FEATURES + [TARGET]].nunique(),
    "pct_missing": (col[FEATURES + [TARGET]].isna().mean() * 100).round(2),
})
overview

In [ ]:
col[NUM_FEATURES + BIN_FEATURES].describe().T.round(2)

## 3. Missing values

Note: road configuration (`C_RCFG`) has no missing values here, because a collision can only be flagged as an
intersection when its road configuration is known.

In [ ]:
miss = (col[FEATURES + [TARGET]].isna().mean() * 100).sort_values()
ax = miss.plot.barh(figsize=(7, 5), color="steelblue")
ax.set_xlabel("% missing")
ax.set_title("Missing values per feature (collision level)")
for i, v in enumerate(miss):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Rows left if every row with any missing feature/target is dropped
complete = col.dropna(subset=FEATURES + [TARGET])
print(f"Complete cases: {len(complete):,} of {len(col):,} ({len(complete)/len(col)*100:.1f}%)")
print("Project minimum is 50,000 records.")

In [ ]:
# Is missingness related to the target? If yes, an 'Unknown' category may carry information.
d = col.dropna(subset=[TARGET])
rows = []
for f in FEATURES:
    m = d[f].isna()
    if m.sum() == 0:
        continue
    rows.append({
        "feature": f,
        "n_missing": int(m.sum()),
        "% 3+ when missing": round((d.loc[m, TARGET] == "3+").mean() * 100, 2),
        "% 3+ when present": round((d.loc[~m, TARGET] == "3+").mean() * 100, 2),
    })
if rows:
    display(pd.DataFrame(rows).sort_values("n_missing", ascending=False))
else:
    print("None of the selected features has missing values.")

## 4. Target distribution (class imbalance)

In [ ]:
print("Rows with no class (0 known casualties):", col[TARGET].isna().sum())
counts = col[TARGET].value_counts().reindex(CLASS_ORDER)
pct = counts / counts.sum() * 100

ax = counts.plot.bar(figsize=(5, 4), color=["#4C72B0", "#DD8452", "#C44E52"], rot=0)
ax.set_xlabel("Casualties per collision"); ax.set_ylabel("Collisions")
ax.set_title("Casualty_Class distribution")
for i, (c, p) in enumerate(zip(counts, pct)):
    ax.text(i, c, f"{c:,}\n({p:.1f}%)", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, counts.max() * 1.2)
plt.tight_layout(); plt.show()

print(f"Imbalance ratio (largest / smallest class): {counts.max() / counts.min():.1f}")

## 5. Feature distributions

In [ ]:
# Numeric / time features
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
col["C_HOUR"].plot.hist(bins=24, ax=axes[0, 0], color="steelblue", edgecolor="white")
axes[0, 0].set_title("Hour of day"); axes[0, 0].set_xlabel("Hour")

col["C_WDAY"].value_counts().sort_index().plot.bar(ax=axes[0, 1], color="steelblue", rot=0)
axes[0, 1].set_title("Day of week (1 = Monday)")

col["C_MNTH"].value_counts().sort_index().plot.bar(ax=axes[1, 0], color="steelblue", rot=0)
axes[1, 0].set_title("Month")

col["C_VEHS"].clip(upper=8).value_counts().sort_index().plot.bar(ax=axes[1, 1], color="steelblue", rot=0)
axes[1, 1].set_title("Vehicles involved (8 = 8 or more)")
plt.tight_layout(); plt.show()

In [ ]:
# Categorical features (top 10 categories each, including Missing)
fig, axes = plt.subplots(3, 2, figsize=(13, 13))
for ax, f in zip(axes.ravel(), CAT_FEATURES):
    vc = label(col[f], f).value_counts(normalize=True).head(10) * 100
    vc.sort_values().plot.barh(ax=ax, color="steelblue")
    ax.set_title(f); ax.set_xlabel("% of collisions")
plt.tight_layout(); plt.show()

In [ ]:
# Binary features
(col[BIN_FEATURES].mean() * 100).round(2).rename("% of collisions = 1").to_frame()

## 6. Outlier check (numeric features)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(x=col["C_VEHS"], ax=axes[0], color="lightsteelblue")
axes[0].set_title("Vehicles involved (all collisions)")

sns.boxplot(data=col.dropna(subset=[TARGET]), x=TARGET, y="C_VEHS",
            order=CLASS_ORDER, ax=axes[1], color="lightsteelblue")
axes[1].set_title("Vehicles involved by Casualty_Class")
plt.tight_layout(); plt.show()

In [ ]:
q1, q3 = col["C_VEHS"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
out = col["C_VEHS"] > upper
print(f"IQR upper fence: {upper:.1f} vehicles")
print(f"Collisions above fence: {out.sum():,} ({out.mean()*100:.2f}%)")
print("\nLargest vehicle counts:")
print(col["C_VEHS"].value_counts().sort_index().tail(10))
print("\nClass mix of outlier collisions (%):")
print((col.loc[out, TARGET].value_counts(normalize=True) * 100).round(1))

## 7. Feature vs target relationships

In [ ]:
def stacked_by_class(f, top=10, ax=None):
    """100% stacked bars: class mix within each category (most common categories)."""
    d = col.dropna(subset=[TARGET]).copy()
    d[f] = label(d[f], f) if f in LABELS else d[f].astype(str)
    keep = d[f].value_counts().head(top).index
    tab = pd.crosstab(d.loc[d[f].isin(keep), f], d[TARGET], normalize="index") * 100
    tab = tab.reindex(columns=CLASS_ORDER).sort_values("3+")   # highest 3+ share on top
    n = d[f].value_counts()
    tab.index = [f"{c} (n={n[c]:,})" for c in tab.index]   # show sample size per category
    tab.plot.barh(stacked=True, ax=ax, color=["#4C72B0", "#DD8452", "#C44E52"],
                  width=0.8, legend=False)
    ax.set_title(f"Casualty_Class mix by {f}"); ax.set_xlabel("%"); ax.set_ylabel("")

fig, axes = plt.subplots(3, 2, figsize=(14, 15))
for ax, f in zip(axes.ravel(), CAT_FEATURES):
    stacked_by_class(f, ax=ax)
handles, labels_ = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels_, title="Casualties", loc="upper center", ncol=3)
plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()

In [ ]:
# % of collisions with 3+ casualties, by binary feature
d = col.dropna(subset=[TARGET])
rows = []
for f in BIN_FEATURES:
    for v in [0, 1]:
        s = d[d[f] == v]
        rows.append({"feature": f, "value": v, "n": len(s),
                     "% 3+": round((s[TARGET] == "3+").mean() * 100, 2)})
pd.DataFrame(rows)

In [ ]:
# % 3+ by hour of day
d = col.dropna(subset=[TARGET, "C_HOUR"])
rate = d.groupby("C_HOUR")[TARGET].apply(lambda s: (s == "3+").mean() * 100)
ax = rate.plot(marker="o", figsize=(8, 3.5))
ax.set_xlabel("Hour"); ax.set_ylabel("% 3+ casualties")
ax.set_title("Share of 3+ casualty collisions by hour")
plt.tight_layout(); plt.show()

## 8. Correlation heatmaps

Most features are categorical, so Pearson correlation is not meaningful for them.
**Cramér's V** (0 = no association, 1 = perfect association) is used for all feature pairs and the target.
Numeric features are binned/treated as categories for this. A Spearman heatmap is added for the numeric/binary features.

In [ ]:
def cramers_v(x, y):
    """Bias-corrected Cramér's V between two categorical series."""
    tab = pd.crosstab(x, y)
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return np.nan
    chi2 = chi2_contingency(tab, correction=False)[0]
    n = tab.values.sum()
    phi2 = chi2 / n
    r, k = tab.shape
    phi2c = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2c / max(1e-12, min(kc - 1, rc - 1)))

cv_df = col[FEATURES + [TARGET]].copy()
cv_df["C_VEHS"] = cv_df["C_VEHS"].clip(upper=6)   # 6 = 6 or more
cv_cols = FEATURES + [TARGET]

cv = pd.DataFrame(index=cv_cols, columns=cv_cols, dtype=float)
for i, a in enumerate(cv_cols):
    for b in cv_cols[i:]:
        sub = cv_df[[a, b]].dropna()
        v = 1.0 if a == b else cramers_v(sub[a].astype(str), sub[b].astype(str))
        cv.loc[a, b] = cv.loc[b, a] = v

plt.figure(figsize=(10, 8))
sns.heatmap(cv, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, square=True,
            cbar_kws={"label": "Cramér's V"}, annot_kws={"size": 7})
plt.title("Cramér's V: association between features and target")
plt.tight_layout(); plt.show()

In [ ]:
# Strongest feature-feature pairs (possible redundancy) and feature-target associations
pairs = (cv.loc[FEATURES, FEATURES].where(np.triu(np.ones((len(FEATURES),) * 2), 1).astype(bool))
         .stack().sort_values(ascending=False))
print("Top feature pairs:\n", pairs.head(8).round(3))
print("\nAssociation with target:\n", cv.loc[FEATURES, TARGET].sort_values(ascending=False).round(3))

In [ ]:
sp_cols = NUM_FEATURES + BIN_FEATURES
plt.figure(figsize=(6.5, 5))
sns.heatmap(col[sp_cols].corr(method="spearman"), annot=True, fmt=".2f",
            cmap="RdBu_r", vmin=-1, vmax=1, square=True, annot_kws={"size": 8})
plt.title("Spearman correlation (numeric and binary features)")
plt.tight_layout(); plt.show()

## Notes for the preprocessing step
Things to look at in the outputs above before deciding on preprocessing:
- Missing %: which features are small enough to drop/impute vs. large enough to keep as "Unknown"
- Whether missingness is linked to the target (section 3 table)
- Complete-case count vs. the 50,000 minimum
- Vehicle-count outliers: errors or real pileups?
- Class imbalance ratio (affects stratified split and class weights)
- Highly associated feature pairs (possible redundancy)


---

# 03 — Data preprocessing

*Source: `03_preprocessing (2)(1).ipynb`*

# Step 3: Data Preprocessing

**Input:** in-memory collision-level data from Step 1
**Output:**
- `ncdb_clean.csv`: cleaned collision table with a `split` column (train/test)
- `X_train.csv.gz`, `X_test.csv.gz`, `y_train.csv`, `y_test.csv`: encoded and scaled, all features
- `X_train_selected.csv.gz`, `X_test_selected.csv.gz`: same, only the selected features

**Sections**
0. Settings (all preprocessing choices in one place)
1. Load and filter
2. Feature engineering (hour periods)
3. Outliers
4. Missing values
5. Train/test split (stratified)
6. Rare categories
7. Encoding and scaling
8. Feature selection (mutual information)
9. Dimensionality reduction check (PCA)
10. Class imbalance (class weights)
11. Save
12. Summary

## 0. Settings

In [ ]:
INTERSECTION_ONLY = True      # True = keep only intersection collisions (matches project title)
VEH_CAP = 6                   # cap vehicle count: 6 means "6 or more"
MISSING_DROP_PCT = 5.0        # feature missing < this %: drop those rows; otherwise "Unknown" category
RARE_PCT = 1.0                # categories under this % of training rows are merged into "Other"
TEST_SIZE = 0.2               # 80/20 split
N_KEEP = 10                   # number of features kept by feature selection (out of 13)
PCA_VARIANCE = 0.90           # PCA check: components needed to explain this much variance
RANDOM_STATE = 42

## 1. Load and filter

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

# Start from the unmodified full collision dataset, not the EDA subset.
col = aggregated_collisions.copy()
log = []   # keeps track of how many rows each step removes
log.append(("Loaded", len(col)))
print("Loaded:", col.shape)

In [ ]:
# Drop collisions with no class (0 known casualties): there is no answer to learn from
col = col[col["Casualty_Class"].notna()].copy()
col["Casualty_Class"] = col["Casualty_Class"].astype(str)
log.append(("Dropped rows with no Casualty_Class", len(col)))

if INTERSECTION_ONLY:
    col = col[col["Is_Intersection"] == 1].copy()
    log.append(("Kept intersection collisions only", len(col)))

print(pd.DataFrame(log, columns=["step", "rows"]))

## 2. Feature engineering

Hour is turned into 4 time periods. As a plain number, the model would think 23:00 and 00:00 are far apart;
as 24 categories, it would add many columns. Day of week and month are treated as categories.

Readable labels replace the NCDB codes so feature names make sense later (check them against the data dictionary).

In [ ]:
LABELS = {
    "C_WTHR": {"1": "Clear", "2": "Overcast", "3": "Rain", "4": "Snow",
               "5": "Freezing rain/hail", "6": "Low visibility", "7": "Strong wind", "Q": "Other"},
    "C_RSUR": {"1": "Dry", "2": "Wet", "3": "Snow", "4": "Slush", "5": "Icy",
               "6": "Sand/gravel", "7": "Muddy", "8": "Oil", "9": "Flooded", "Q": "Other"},
    "C_RALN": {"1": "Straight level", "2": "Straight gradient", "3": "Curved level",
               "4": "Curved gradient", "5": "Top of hill", "6": "Bottom of hill", "Q": "Other"},
    "C_RCFG": {"01": "Non-intersection", "02": "Intersection", "03": "Driveway/parking",
               "04": "Railroad crossing", "05": "Bridge/overpass", "06": "Tunnel",
               "07": "Passing lane", "08": "Ramp", "09": "Traffic circle",
               "10": "Express lane", "11": "Collector lane", "12": "Transfer lane", "QQ": "Other"},
    "C_TRAF": {"01": "Signals working", "02": "Signals flashing", "03": "Stop sign",
               "04": "Yield sign", "05": "Warning sign", "06": "Ped crosswalk",
               "07": "Police officer", "08": "School guard", "09": "School crossing",
               "10": "Reduced speed zone", "11": "No passing zone", "12": "Road markings",
               "13": "School bus", "15": "Rail signals/gates", "16": "Rail signs only",
               "17": "Not specified", "18": "No control", "QQ": "Other"},
    "C_CONF": {"01": "Hit moving object", "02": "Hit stationary object", "03": "Ran off left",
               "04": "Ran off right", "05": "Rollover", "06": "Other single vehicle",
               "21": "Rear-end", "22": "Side swipe", "23": "Left turn conflict same dir",
               "24": "Right turn conflict same dir", "25": "Other same direction",
               "31": "Head-on", "32": "Approaching side swipe", "33": "Left turn across traffic",
               "34": "Right turn", "35": "Right angle", "36": "Other different direction",
               "41": "Hit parked vehicle", "QQ": "Other"},
}
DAYS = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}

for c, mapping in LABELS.items():
    col[c] = col[c].map(lambda v: mapping.get(v, v) if pd.notna(v) else np.nan)

def hour_period(h):
    if pd.isna(h): return np.nan
    if h < 6:  return "Night (0-5)"
    if h < 12: return "Morning (6-11)"
    if h < 18: return "Afternoon (12-17)"
    return "Evening (18-23)"

col["Hour_Period"] = col["C_HOUR"].apply(hour_period)
col["Weekday"] = col["C_WDAY"].map(DAYS)
col["Month"] = col["C_MNTH"].map(lambda m: f"M{int(m):02d}" if pd.notna(m) else np.nan)

col["Hour_Period"].value_counts(dropna=False)

In [ ]:
CAT_FEATURES = ["C_RCFG", "C_TRAF", "C_WTHR", "C_RSUR", "C_RALN", "C_CONF",
                "Hour_Period", "Weekday", "Month"]
NUM_FEATURES = ["C_VEHS"]
BIN_FEATURES = ["Heavy_Vehicle_Present", "Cyclist_Present", "Safety_Device_Not_Used"]
FEATURES = CAT_FEATURES + NUM_FEATURES + BIN_FEATURES   # 13 features
TARGET = "Casualty_Class"
CLASS_ORDER = ["1", "2", "3+"]

# Leakage columns (built from the target) are never used as inputs
LEAKAGE = ["Total_Injuries", "Total_Fatalities", "Total_Casualties", "Fatal_Collision", "C_SEV"]
print(len(FEATURES), "features")

## 3. Outliers

Vehicle count is the only numeric feature where outliers matter. Large pileups are real events and are
often 3+ casualty collisions (the smallest class), so they are **capped** at `VEH_CAP` instead of removed.
Further outlier analysis (Isolation Forest) is planned for the final project.

In [ ]:
q1, q3 = col["C_VEHS"].quantile([0.25, 0.75])
fence = q3 + 1.5 * (q3 - q1)
above_fence = col["C_VEHS"] > fence
print(f"IQR upper fence: {fence:.1f} vehicles -> {above_fence.sum():,} collisions above it")
print("Class mix above fence (%):")
print((col.loc[above_fence, TARGET].value_counts(normalize=True) * 100).round(1))

capped = (col["C_VEHS"] > VEH_CAP).sum()
col["C_VEHS"] = col["C_VEHS"].clip(upper=VEH_CAP)
print(f"\nCapped {capped:,} collisions at {VEH_CAP} vehicles")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
sns.boxplot(x=col["C_VEHS"], ax=axes[0], color="lightsteelblue")
axes[0].set_title(f"Vehicles after capping at {VEH_CAP}")
sns.boxplot(data=col, x=TARGET, y="C_VEHS", order=CLASS_ORDER, ax=axes[1], color="lightsteelblue")
axes[1].set_title("Vehicles by Casualty_Class (after capping)")
plt.tight_layout(); plt.show()

## 4. Missing values

Rule applied to every feature:
- **Missing < `MISSING_DROP_PCT`%** -> drop those rows (small loss, no guessing)
- **Missing >= `MISSING_DROP_PCT`%** -> categorical: new **"Unknown"** category; binary: becomes a 3-level category (Yes/No/Unknown); numeric: median (fitted on training data only)

Keeping "Unknown" avoids losing many rows, and in the NCDB missing often means a province does not report
that field, so it can carry information. Dropping rows before the split is fine; median filling happens after the split.

In [ ]:
miss_pct = col[FEATURES].isna().mean() * 100
decision = pd.DataFrame({"pct_missing": miss_pct.round(2)})
# Use the exact %, not the rounded one, so even a handful of missing values gets handled
decision["action"] = np.where(miss_pct == 0, "none",
                      np.where(miss_pct < MISSING_DROP_PCT, "drop rows", "Unknown / median"))
decision.sort_values("pct_missing", ascending=False)

In [ ]:
drop_cols = decision.index[decision["action"] == "drop rows"].tolist()
before = len(col)
col = col.dropna(subset=drop_cols)
log.append((f"Dropped rows missing in: {', '.join(drop_cols) or 'none'}", len(col)))
print(f"Dropped {before - len(col):,} rows ({(before - len(col)) / before * 100:.2f}%)")

unknown_cols = decision.index[decision["action"] == "Unknown / median"].tolist()
for c in unknown_cols:
    if c in CAT_FEATURES:
        col[c] = col[c].fillna("Unknown")
    elif c in BIN_FEATURES:
        col[c] = col[c].map({1: "Yes", 0: "No", 1.0: "Yes", 0.0: "No"}).fillna("Unknown")
        BIN_FEATURES.remove(c); CAT_FEATURES.append(c)   # now treated as a category
# Numeric features with many missing values are median-filled after the split (section 5)
MEDIAN_FILL = [c for c in unknown_cols if c in NUM_FEATURES]

print("Became 'Unknown' category:", [c for c in unknown_cols if c not in MEDIAN_FILL] or "none")
print("Median fill after split:", MEDIAN_FILL or "none")
print("Rows now:", f"{len(col):,}")

## 5. Train/test split (stratified)

Stratified so each set keeps the same class mix (important because 3+ is rare).
Everything learned from data (rare categories, medians, scaling, encoding, feature selection) is fitted on
the **training set only**, so no test information leaks into training.

In [ ]:
train_df, test_df = train_test_split(col, test_size=TEST_SIZE, stratify=col[TARGET],
                                     random_state=RANDOM_STATE)
train_df, test_df = train_df.copy(), test_df.copy()

for c in MEDIAN_FILL:
    med = train_df[c].median()
    train_df[c] = train_df[c].fillna(med); test_df[c] = test_df[c].fillna(med)

split_mix = pd.DataFrame({
    "train %": train_df[TARGET].value_counts(normalize=True).reindex(CLASS_ORDER) * 100,
    "test %": test_df[TARGET].value_counts(normalize=True).reindex(CLASS_ORDER) * 100,
}).round(2)
print(f"Train: {len(train_df):,}   Test: {len(test_df):,}")
split_mix

## 6. Rare categories

Categories with fewer than `RARE_PCT`% of training rows give the model very few examples, and they add extra
columns after one-hot encoding. They are merged into **"Other"** (decided on training data, applied to both sets).

In [ ]:
rare_report = []
for c in CAT_FEATURES:
    freq = train_df[c].value_counts(normalize=True) * 100
    rare = freq[freq < RARE_PCT].index.tolist()
    if rare:
        train_df[c] = train_df[c].where(~train_df[c].isin(rare), "Other")
    keep = set(train_df[c].unique())
    test_df[c] = test_df[c].where(test_df[c].isin(keep), "Other")
    rare_report.append({"feature": c, "categories before": len(freq),
                        "merged into Other": len(rare), "categories after": train_df[c].nunique()})
pd.DataFrame(rare_report)

## 7. Encoding and scaling

- **Categorical** -> one-hot encoding (one 0/1 column per category). No order is implied, which fits nominal
  codes like weather or collision type.
- **Numeric** (vehicle count) -> StandardScaler (mean 0, standard deviation 1), fitted on training data.
  Mainly matters for logistic regression and k-means; tree models are not affected by scale.
- **Binary** features are already 0/1 and are passed through unchanged.

In [ ]:
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
    ("num", StandardScaler(), NUM_FEATURES),
    ("bin", "passthrough", BIN_FEATURES),
], verbose_feature_names_out=False)

X_train = pd.DataFrame(pre.fit_transform(train_df[FEATURES]),
                       columns=pre.get_feature_names_out(), index=train_df.index)
X_test = pd.DataFrame(pre.transform(test_df[FEATURES]),
                      columns=pre.get_feature_names_out(), index=test_df.index)
y_train, y_test = train_df[TARGET], test_df[TARGET]
assert not X_train.isna().any().any() and not X_test.isna().any().any(), "NaN left after preprocessing"

print(f"Columns before encoding: {len(FEATURES)}   after encoding: {X_train.shape[1]}")
print("Vehicle count after scaling (train): mean "
      f"{X_train['C_VEHS'].mean():.2f}, std {X_train['C_VEHS'].std():.2f}")
X_train.head()

## 8. Feature selection: mutual information

Mutual information (MI) measures how much knowing a feature reduces uncertainty about the casualty class
(0 = no information). It is a **filter method**, independent of any classification model, and works for
categorical features and non-linear relationships. It is computed per original feature on the training set.
The top `N_KEEP` features are kept; the final project will compare models with and without this selection.

In [ ]:
enc = OrdinalEncoder()
X_mi = enc.fit_transform(train_df[FEATURES].astype(str))
mi = mutual_info_classif(X_mi, y_train, discrete_features=True, random_state=RANDOM_STATE)
mi = pd.Series(mi, index=FEATURES).sort_values(ascending=False)

selected = mi.head(N_KEEP).index.tolist()
dropped = mi.index[N_KEEP:].tolist()

colors = ["#4C72B0" if f in selected else "#BBBBBB" for f in mi.index]
ax = mi.sort_values().plot.barh(figsize=(7, 5), color=colors[::-1])
ax.set_xlabel("Mutual information with Casualty_Class")
ax.set_title(f"Feature selection: top {N_KEEP} kept (blue), dropped (grey)")
plt.tight_layout(); plt.show()

print("Selected:", selected)
print("Dropped: ", dropped)
mi.round(5).to_frame("MI")

In [ ]:
# Encoded columns that belong to the selected features
def belongs(colname, feats):
    return any(colname == f or colname.startswith(f + "_") for f in feats)

sel_cols = [c for c in X_train.columns if belongs(c, selected)]
X_train_sel, X_test_sel = X_train[sel_cols], X_test[sel_cols]
print(f"Encoded columns: all features {X_train.shape[1]} -> selected features {X_train_sel.shape[1]}")

## 9. Dimensionality reduction check (PCA)

PCA is only needed if the data is high-dimensional. This checks how many components are needed to keep
`PCA_VARIANCE` of the variance. PCA is **not applied** to the saved data by default because it turns features
into combinations that are hard to interpret (we want to know which factors matter). It will be used in the
final project to visualize clusters in 2D.

In [ ]:
pca = PCA(random_state=RANDOM_STATE).fit(X_train)
cum = np.cumsum(pca.explained_variance_ratio_)
n_needed = int(np.argmax(cum >= PCA_VARIANCE) + 1)

ax = plt.figure(figsize=(7, 3.5)).gca()
ax.plot(range(1, len(cum) + 1), cum, marker=".")
ax.axhline(PCA_VARIANCE, color="red", ls="--", lw=1)
ax.axvline(n_needed, color="red", ls="--", lw=1)
ax.set_xlabel("Number of components"); ax.set_ylabel("Cumulative explained variance")
ax.set_title("PCA explained variance (training set)")
plt.tight_layout(); plt.show()

print(f"Encoded columns: {X_train.shape[1]}")
print(f"Components for {PCA_VARIANCE:.0%} variance: {n_needed} "
      f"({n_needed / X_train.shape[1]:.0%} of columns)")

## 10. Class imbalance

The 3+ class is rare. Plan: **stratified split** (done above) + **class weights** in the models, so mistakes on
rare classes cost more. Resampling such as SMOTE can be compared in the final project, applied to the training
set only. The weights below are computed now so they can be reused.

In [ ]:
weights = compute_class_weight("balanced", classes=np.array(CLASS_ORDER), y=y_train)
class_weights = {c: round(float(w), 3) for c, w in zip(CLASS_ORDER, weights)}
print("Training class counts:", y_train.value_counts().reindex(CLASS_ORDER).to_dict())
print("Balanced class weights:", class_weights)

## 11. Save

In [ ]:
clean = pd.concat([train_df.assign(split="train"), test_df.assign(split="test")])
clean_cols = ["C_CASE", "C_YEAR"] + FEATURES + [TARGET, "Is_Intersection"] + \
             [c for c in LEAKAGE if c in clean.columns] + ["split"]
clean[clean_cols].to_csv(OUTPUT_DIR / "ncdb_clean.csv", index=False)

X_train.to_csv(OUTPUT_DIR / "X_train.csv.gz", index=False)
X_test.to_csv(OUTPUT_DIR / "X_test.csv.gz", index=False)
X_train_sel.to_csv(OUTPUT_DIR / "X_train_selected.csv.gz", index=False)
X_test_sel.to_csv(OUTPUT_DIR / "X_test_selected.csv.gz", index=False)
y_train.to_csv(OUTPUT_DIR / "y_train.csv", index=False)
y_test.to_csv(OUTPUT_DIR / "y_test.csv", index=False)

out_files = ["ncdb_clean.csv", "X_train.csv.gz", "X_test.csv.gz", "X_train_selected.csv.gz",
             "X_test_selected.csv.gz", "y_train.csv", "y_test.csv"]
for f in out_files:
    print(f"{f:28s} {(OUTPUT_DIR / f).stat().st_size / 1e6:6.2f} MB")

## 12. Summary

In [ ]:
print("Row flow:")
print(pd.DataFrame(log, columns=["step", "rows"]).to_string(index=False))
print(f"\nTrain / test: {len(train_df):,} / {len(test_df):,} (stratified {1-TEST_SIZE:.0%}/{TEST_SIZE:.0%})")
print(f"Vehicle count capped at {VEH_CAP}; standardized")
print("'Unknown' category used for:", [c for c in unknown_cols if c not in MEDIAN_FILL] or "none")
print(f"Rare categories (<{RARE_PCT}%) merged into 'Other'")
print(f"One-hot encoded columns: {X_train.shape[1]} (selected: {X_train_sel.shape[1]})")
print(f"Selected features ({N_KEEP}):", selected)
print(f"PCA: {n_needed} components for {PCA_VARIANCE:.0%} variance (not applied)")
print("Class weights:", class_weights)

## Finished — collect all results

This last cell creates a ZIP containing the collision-level dataset and the seven preprocessing output files. The ZIP and CSV files are saved in `outputs/`. If you are running Colab, you can enable the `AUTO_DOWNLOAD_ZIP_IN_COLAB` setting at the top to receive one ZIP download automatically.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display

ZIP_FILE = OUTPUT_DIR / "ncdb_full_pipeline_outputs.zip"
with ZipFile(ZIP_FILE, mode="w", compression=ZIP_DEFLATED) as archive:
    for path in [AGGREGATED_FILE] + [OUTPUT_DIR / name for name in out_files]:
        archive.write(path, arcname=path.name)
print("✓ All three stages finished successfully.")
print("Files saved to:", OUTPUT_DIR.resolve())
print("Results ZIP:", ZIP_FILE.resolve())
display(FileLink(str(ZIP_FILE)))
if AUTO_DOWNLOAD_ZIP_IN_COLAB:
    try:
        from google.colab import files
        files.download(str(ZIP_FILE))
    except ImportError:
        print("Colab download skipped: this environment is not Google Colab.")
